# Skenario A: Visualisasi REST vs gRPC

Notebook ini membaca `results/scenario_a/raw.csv` langsung dari GitHub (jadi Anda cukup **Run all**
tiap kali data baru sudah di-push, tidak perlu upload manual) dan menghasilkan 4 figure: **Latency**,
**Throughput**, **CPU**, dan **RAM**, masing-masing dipecah per segmen komunikasi.

Setiap panel memakai gaya P50 garis solid, P99 garis putus-putus, dengan area terarsir di antaranya
supaya rentang persentil kelihatan sekali pandang. Warna REST/gRPC sudah divalidasi CVD-safe
(`node scripts/validate_palette.js` dari skill dataviz, ΔE 74.6, jauh di atas ambang 12).

Mau ubah judul, warna, atau ukuran figure? Semuanya ada di sel **Konfigurasi** di bawah. Sel-sel
setelahnya tidak perlu disentuh.

## Konfigurasi

Ubah nilai di sel ini sesuai kebutuhan, lalu Run all.

In [ ]:
# ==== Konfigurasi: ubah di sini ====

# URL raw.csv di GitHub (branch dev). Kalau gagal diakses (mis. belum di-push,
# atau repo private), notebook otomatis fallback ke file lokal di bawah.
GITHUB_RAW_CSV_URL = (
    "https://raw.githubusercontent.com/afifksupriyadi/"
    "grpc-rest-benchmark-video-transcoder/dev/results/scenario_a/raw.csv"
)
LOCAL_CSV_FALLBACK = "../results/scenario_a/raw.csv"

# Warna REST/gRPC, palet biru/merah muted (gaya seaborn "deep").
# Konsisten dengan warna yang sudah dipakai di tabel HTML (scripts/collect_scenario_a.py)
# secara hue, meski di sini sengaja lebih muted untuk kesan cetak/laporan.
COLORS = {
    "rest": "#4C72B0",
    "grpc": "#C44E52",
}
PROTOCOL_LABEL = {"rest": "REST", "grpc": "gRPC"}

TITLES = {
    "latency_seconds": "Skenario A: LATENCY REST vs gRPC",
    "throughput_bytes_per_second": "Skenario A: THROUGHPUT REST vs gRPC",
    "cpu_usage_ratio": "Skenario A: CPU REST vs gRPC",
    "memory_usage_bytes": "Skenario A: RAM REST vs gRPC",
}

# Maksimal 2 panel per baris (supaya muat dipaste ke Word tanpa perlu di-crop).
# Ukuran per panel yang tetap, BUKAN lebar total tetap. Jumlah baris menyesuaikan
# otomatis: 4 segmen jadi grid 2x2, 3 segmen jadi 2+1 (slot terakhir disembunyikan).
MAX_COLS = 2
PANEL_SIZE = (6.5, 4.5)  # (lebar, tinggi) per panel, dalam inci
FONT_SIZES = {
    "suptitle": 16,
    "subplot_title": 13,
    "axis_label": 10,
    "tick_label": 9,
    "legend": 11,
}
OUTPUT_DIR = "figures"  # folder PNG hasil export, dibuat relatif ke lokasi notebook
DPI = 150

## Setup (tidak perlu diubah)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use("seaborn-v0_8-whitegrid")

try:
    df = pd.read_csv(GITHUB_RAW_CSV_URL)
    print(f"Data dimuat dari GitHub: {len(df)} baris")
except Exception as e:
    print(f"Gagal fetch dari GitHub ({e}), pakai file lokal: {LOCAL_CSV_FALLBACK}")
    df = pd.read_csv(LOCAL_CSV_FALLBACK)
    print(f"Data dimuat dari lokal: {len(df)} baris")

df["payload_mb"] = df["payload_size"].str.replace("mb", "", case=False, regex=False).astype(int)
df = df.sort_values("payload_mb")
df.head()

In [ ]:
SEGMENT_LABEL = {
    "client_to_gateway": "Segmen 1: Client ke Gateway",
    "gateway_to_worker": "Segmen 2: Gateway ke Worker",
    "worker_to_gateway": "Segmen 3: Worker ke Gateway",
    "gateway_to_client": "Segmen 4: Gateway ke Client",
}
SEGMENT_ORDER = list(SEGMENT_LABEL.keys())


def plot_metric_figure(metric, title):
    """One figure, one subplot per segment that actually has data for this
    metric (CPU/RAM naturally drop 'Worker ke Gateway'; see architecture note
    in the generated HTML tables). P50 solid + P99 dashed + shaded band,
    REST/gRPC as the only two colors, legend shared once for the whole figure.
    Grid is capped at MAX_COLS columns (wraps to a new row) so the exported
    PNG still fits a Word page without needing to shrink it illegibly."""
    sub = df[df["metric"] == metric]
    segments = [s for s in SEGMENT_ORDER if s in sub["segment"].unique()]
    unit = sub["unit"].mode().iat[0]

    n_cols = min(MAX_COLS, len(segments))
    n_rows = -(-len(segments) // n_cols)  # ceil division

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(PANEL_SIZE[0] * n_cols, PANEL_SIZE[1] * n_rows),
        squeeze=False,
    )
    axes_flat = axes.flatten()

    legend_handles = {}
    for i, seg in enumerate(segments):
        ax = axes_flat[i]
        seg_df = sub[sub["segment"] == seg]
        for protocol, color in COLORS.items():
            pdf = seg_df[seg_df["protocol"] == protocol].sort_values("payload_mb")
            p50 = pdf[pdf["percentile"] == 50]
            p99 = pdf[pdf["percentile"] == 99]
            if p50.empty:
                continue
            label = f"{PROTOCOL_LABEL[protocol]} P50"
            (line,) = ax.plot(
                p50["payload_mb"], p50["value"],
                color=color, lw=2, marker="o", markersize=7, label=label,
            )
            legend_handles[label] = line
            if not p99.empty:
                label99 = f"{PROTOCOL_LABEL[protocol]} P99"
                (line99,) = ax.plot(
                    p99["payload_mb"], p99["value"],
                    color=color, lw=1.5, ls="--", marker="s", markersize=6, label=label99,
                )
                legend_handles[label99] = line99
                ax.fill_between(
                    p50["payload_mb"], p50["value"], p99["value"],
                    color=color, alpha=0.08,
                )

        ax.set_title(SEGMENT_LABEL[seg], fontsize=FONT_SIZES["subplot_title"], fontweight="bold")
        ax.set_xlabel("Ukuran Payload (MB)", fontsize=FONT_SIZES["axis_label"])
        if i % n_cols == 0:
            ax.set_ylabel(unit, fontsize=FONT_SIZES["axis_label"])
        ax.set_xticks(sorted(df["payload_mb"].unique()))
        ax.tick_params(labelsize=FONT_SIZES["tick_label"])

    # hide leftover slots when segments don't fill the last row completely
    for j in range(len(segments), len(axes_flat)):
        fig.delaxes(axes_flat[j])

    fig.legend(
        legend_handles.values(), legend_handles.keys(),
        loc="lower center", ncol=len(legend_handles), frameon=False,
        bbox_to_anchor=(0.5, -0.02 / n_rows), fontsize=FONT_SIZES["legend"],
    )
    fig.suptitle(title, fontsize=FONT_SIZES["suptitle"], fontweight="bold", y=1.0 + 0.02 / n_rows)
    fig.tight_layout()
    return fig

## Latency

In [ ]:
Path(OUTPUT_DIR).mkdir(exist_ok=True)
fig = plot_metric_figure("latency_seconds", TITLES["latency_seconds"])
fig.savefig(f"{OUTPUT_DIR}/scenario_a_latency.png", dpi=DPI, bbox_inches="tight")
plt.show()

## Throughput

In [ ]:
fig = plot_metric_figure("throughput_bytes_per_second", TITLES["throughput_bytes_per_second"])
fig.savefig(f"{OUTPUT_DIR}/scenario_a_throughput.png", dpi=DPI, bbox_inches="tight")
plt.show()

## CPU

In [ ]:
fig = plot_metric_figure("cpu_usage_ratio", TITLES["cpu_usage_ratio"])
fig.savefig(f"{OUTPUT_DIR}/scenario_a_cpu.png", dpi=DPI, bbox_inches="tight")
plt.show()

## RAM

In [ ]:
fig = plot_metric_figure("memory_usage_bytes", TITLES["memory_usage_bytes"])
fig.savefig(f"{OUTPUT_DIR}/scenario_a_ram.png", dpi=DPI, bbox_inches="tight")
plt.show()

---
Selesai. 4 file PNG ada di folder `figures/` (di Colab: klik ikon folder di sidebar kiri untuk
download). Ubah warna/judul di sel **Konfigurasi** paling atas lalu **Run all** lagi kalau mau
tweak tanpa sentuh kode plotting.